In [38]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split

In [39]:
file_path = "/kaggle/input/datasets/ahmedelfazouan/belka-enc-dataset/train_enc.parquet"
parquet_file = pq.ParquetFile(file_path)

batch_iter = parquet_file.iter_batches(batch_size=50000)
first_batch = next(batch_iter)

train = first_batch.to_pandas()
train.head()

,enc0,enc1,enc2,enc3,enc4,enc5,enc6,enc7,enc8,enc9,...,enc135,enc136,enc137,enc138,enc139,enc140,enc141,bind1,bind2,bind3
0,8,22,8,8,28,12,27,12,12,12,...,0,0,0,0,0,0,0,0,0,0
1,8,22,8,8,28,12,27,12,12,12,...,0,0,0,0,0,0,0,0,0,0
2,8,22,8,8,28,12,27,12,12,12,...,0,0,0,0,0,0,0,0,0,0
3,8,22,8,8,28,12,27,12,12,12,...,0,0,0,0,0,0,0,0,0,0
4,8,22,8,8,28,12,27,12,12,12,...,0,0,0,0,0,0,0,0,0,0


In [40]:
train.shape
train.columns

Index(['enc0', 'enc1', 'enc2', 'enc3', 'enc4', 'enc5', 'enc6', 'enc7', 'enc8',
       'enc9',
       ...
       'enc135', 'enc136', 'enc137', 'enc138', 'enc139', 'enc140', 'enc141',
       'bind1', 'bind2', 'bind3'],
      dtype='object', length=145)

In [41]:
feature_cols = [col for col in train.columns if col.startswith("enc")]
target_cols = [col for col in train.columns if col.startswith("bind")]

X = train[feature_cols].values
y = train[target_cols].values


X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


X_train = torch.as_tensor(X_train, dtype=torch.long)
X_val = torch.as_tensor(X_val, dtype=torch.long)
y_train = torch.as_tensor(y_train, dtype=torch.float32)
y_val = torch.as_tensor(y_val, dtype=torch.float32)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

In [42]:
print(X_train.shape)
print(y_train.shape)
print(X_train[0])
print(y_train[0])
batch_X, batch_y = next(iter(train_loader))

print(batch_X.shape)
print(batch_y.shape)
print(X_train.min())
print(X_train.max())
print(y_train.sum(dim=0))
for batch_X, batch_y in train_loader:
    print(batch_X.shape, batch_y.shape)
    break

torch.Size([40000, 142])
torch.Size([40000, 3])
tensor([ 8, 22,  8,  8, 29,  8,  3,  3,  5, 32, 17,  8,  8, 17, 26, 28, 19, 33,
        29, 30,  2, 32, 19, 33, 12, 27, 35, 12, 17, 33,  8,  8, 17,  8, 33, 17,
         8, 19,  8, 19, 28,  8,  8, 19, 35, 12, 17, 33, 12, 18, 12, 12, 17,  8,
        17,  8, 19, 17,  8, 19,  8, 19, 29, 35,  5, 32, 35, 18, 19, 35, 27,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])
tensor([0., 0., 0.])
torch.Size([256, 142])
torch.Size([256, 3])
tensor(0)
tensor(36)
tensor([83., 77., 91.])
torch.Size([256, 142]) torch.Size([256, 3])


In [43]:


class LigandCNN(nn.Module):
    def __init__(self, vocab_size=37, embed_dim=64, n_filters=128, output_dim=3, dropout=0.2):
        super().__init__()

        # 0 is padding, so tell embedding to ignore it
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        # parallel conv branches
        self.conv3 = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=n_filters,
            kernel_size=3,
            padding=1
        )
        self.conv5 = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=n_filters,
            kernel_size=5,
            padding=2
        )
        self.conv7 = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=n_filters,
            kernel_size=7,
            padding=3
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filters * 3, output_dim)

    def forward(self, x):
        # x: (batch_size, 142)

        x = self.embedding(x)          # (batch_size, 142, embed_dim)
        x = x.permute(0, 2, 1)         # (batch_size, embed_dim, 142)

        x3 = F.relu(self.conv3(x))     # (batch_size, n_filters, 142)
        x5 = F.relu(self.conv5(x))
        x7 = F.relu(self.conv7(x))

        # global max pooling over sequence length
        x3 = torch.max(x3, dim=2).values   # (batch_size, n_filters)
        x5 = torch.max(x5, dim=2).values
        x7 = torch.max(x7, dim=2).values

        x = torch.cat([x3, x5, x7], dim=1) # (batch_size, n_filters*3)
        x = self.dropout(x)
        out = self.fc(x)                   # (batch_size, 3)

        return out

In [44]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LigandCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [45]:
batch_X, batch_y = next(iter(train_loader))
batch_X = batch_X.to(device)
batch_y = batch_y.to(device)

out = model(batch_X)
print(out.shape)

torch.Size([256, 3])


In [46]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)

            total_loss += loss.item()

    return total_loss / len(loader)

In [47]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1/5 | Train Loss: 0.0273 | Val Loss: 0.0133
Epoch 2/5 | Train Loss: 0.0140 | Val Loss: 0.0121
Epoch 3/5 | Train Loss: 0.0125 | Val Loss: 0.0110
Epoch 4/5 | Train Loss: 0.0113 | Val Loss: 0.0111
Epoch 5/5 | Train Loss: 0.0108 | Val Loss: 0.0103


In [48]:
model.eval()

all_probs = []
all_true = []

with torch.no_grad():
    for batch_X, batch_y in val_loader:
        batch_X = batch_X.to(device)
        logits = model(batch_X)
        probs = torch.sigmoid(logits).cpu()

        all_probs.append(probs)
        all_true.append(batch_y)

all_probs = torch.cat(all_probs, dim=0)
all_true = torch.cat(all_true, dim=0)

print(all_probs.shape)
print(all_true.shape)

torch.Size([10000, 3])
torch.Size([10000, 3])


In [50]:
preds = (all_probs >= 0.5).float()

In [52]:
print("True positives per target:", all_true.sum(dim=0))
print("Predicted positives per target:", preds.sum(dim=0))

True positives per target: tensor([10., 24., 24.])
Predicted positives per target: tensor([0., 0., 0.])
